# Working with Time Series

## Loading Libraries

In [ ]:
# Numerical Computing
import numpy as np

# Data Manipulation 
import pandas as pd

# Data Visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# OS & URL
import os
import urllib.request

### Retrieving Data

In [ ]:
os.makedirs('/Users/joaquinromero/Files/Desktop/EP2/data', exist_ok=True)

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/dirtydevil.txt'

In [ ]:
df = pd.read_csv(url, skiprows=lambda num: num <34 or num == 35,
                 sep='\t')

In [ ]:
def tweak_river(df_):
    return (df_
 .assign(datetime=pd.to_datetime(df_.datetime))
 .rename(columns={'144166_00060': 'cfs',
                  '144167_00065': 'gage_height'})
 .set_index('datetime')
)

In [ ]:
dd = tweak_river(df)
dd

### Adding Time\-zone Information

In [ ]:
dd.tz_cd

In [ ]:
def tweak_river(df_):
    return (df_
      .assign(datetime=lambda df_:
          pd.to_datetime(df_.datetime + " " +
              df_.tz_cd.str.replace('MST', '-0700')
                 .str.replace('MDT', '-0600'),
              format='%Y-%m-%d %H:%M %z', utc=True))
      .rename(columns={'144166_00060': 'cfs',
                       '144167_00065': 'gage_height'})
      .set_index('datetime')
    )

In [ ]:
def to_america_denver_time(df_, time_col, tz_col):
    return (df_
            .assign(**{tz_col: df_[tz_col].replace('MDT', 'MST7MDT')})
            .groupby(tz_col)
            [time_col]
            .transform(lambda s: pd.to_datetime(s)
                .dt.tz_localize(s.name, ambiguous=True)
                .dt.tz_convert('America/Denver'))
           )

In [ ]:
def tweak_river(df_):
    return (df_
      .assign(datetime=to_america_denver_time(df_, 'datetime',
              'tz_cd'))
      .rename(columns={'144166_00060': 'cfs',
                       '144167_00065': 'gage_height'})
      .set_index('datetime')
     )

In [ ]:
print(dd)

### Exploring The Data

In [ ]:
fig, ax = plt.subplots(dpi=600)
dd.cfs.plot(ax=ax)

plt.grid(True)
plt.show()

In [ ]:
dd.cfs.describe()

### Slicing Timer Series

In [ ]:
(dd
.cfs
.loc['2018']
)

In [ ]:
(dd
.cfs
.sort_index()
.loc['2018-03-05':'2019/05']
)

In [ ]:
(dd
.sort_index()
.cfs
.loc['2018/3':'2019/5']
.plot(color='tomato')
)

plt.grid(True)
plt.show()

In [ ]:
(dd
.sort_index()
.cfs
.loc['2018/3':'2019/5']
.clip(upper=400)
.plot(color='gold')
)

plt.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=600, figsize=(8, 4))

dd2018 = (dd
    .sort_index()
    .cfs
    .loc['2018/3':'2019/5']
    .clip(upper=400)
    )

(dd2018
    .resample('D')
    .mean()
    .plot(figsize=(10, 4), alpha=.5, linewidth=1, label='Daily', color='forestgreen')
    )

plt.grid(True)
plt.show()

In [ ]:
ax = (dd2018
    .resample('D')
    .mean()
    .rolling(7)
    .mean()
    .plot(figsize=(10,4), ax=ax, label='7-day Rolling')
    )

ax.legend()
ax.set_title('Dirty Devil Flow 2018 (cfs)')
sns.despine()
# fig.savefig('img/EP2/dd4.png', dpi=600, bbox_inches='tight')
plt.grid(True)
plt.show()

### Missing Timeseries Data

In [ ]:
(dd
  [['cfs']]
  .loc['2018/3':'2019/5']
  .query('cfs.isna()')
)

In [ ]:
(dd[['cfs']]
    .loc['2018/7/7':'2018/7/8']
    .plot(figsize=(10,3))
    )

plt.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=600, figsize=(10,3))   
dd_july = (dd['cfs'].loc['2018/7/7 11:00':'2018/7/7 20:00']
)

dd_july.plot(ax=ax, label='original', linewidth=2)

(dd_july
    .bfill()
    .add(.05)
    .plot(label='bfill', ax=ax, linewidth=.5)
    )

(dd_july
    .ffill()
    .add(.1)
    .plot(label='ffill', ax=ax, linewidth=.5)
    )

(dd_july
    .interpolate(method='polynomial', order=3)
    .add(.15)
    .plot(label='interpolate poly (order 3)', ax=ax, linewidth=.5)
    )

(dd_july
    .interpolate()
    .add(.2)
    .plot(label='interpolate default', ax=ax, linewidth=.5)
    )

(dd_july
    .interpolate(method='nearest')
    .add(.25)
    .plot(label='interpolate nearest', ax=ax, linewidth=.5)
    )

(dd_july
.fillna(1)
.add(.3))
 
ax.legend()   
ax.set_title('Missing Values Demo') 
sns.despine()
plt.grid(True)
plt.show()

### Exploring Seasonality

In [ ]:
print(dd
    .groupby(dd.index.month)
    .cfs
    .describe()
    )

In [ ]:
(dd
    .groupby(dd.index.month)
    .cfs
    .describe()
    )

In [ ]:
fig, ax = plt.subplots(dpi=600, figsize=(10,4)) 

(dd
    .groupby(dd.index.month)
    ['cfs']
    .describe()
    ['mean']
    .plot
    .bar(ax=ax, color='forestgreen')
    )

plt.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=600, figsize=(10,4)) 

(dd
    .groupby(dd.index.month)
    ['cfs']
    .describe()
    .loc[:, 'min':'75%']
    .plot.bar(ax=ax)
    )

plt.grid(True)
plt.show()

In [ ]:
sns.boxplot(
    data=(
        dd
        .reset_index()
        .assign(
            cfs=lambda df_: df_.cfs.clip(upper=400),
            Month=lambda df_: df_.datetime.dt.month
        )
    ),
    x='Month',
    y='cfs'
)

plt.grid(True)
plt.show()

### Resampling Data

In [ ]:
dd.cfs

In [ ]:
print(dd
.resample('D')
.median(numeric_only=True)
)  

### Rules with Offset Aliases

In [ ]:
print(dd
.resample('2D')
.median(numeric_only=True)
)  

### Combining Offset Aliases

In [ ]:
print(dd
.resample('3D2h10min')
.median(numeric_only=True)
)

### Anchored Offset Aliases

In [ ]:
print(dd
.resample('Q-DEC')
.median(numeric_only=True)
)  

In [ ]:
print(dd
.resample('Q-JAN')
.median(numeric_only=True)
)  

### Resampling to Fine\-grain Frequency

In [ ]:
print(dd
.resample('2min')
.median(numeric_only=True)
)  

In [ ]:
print(dd
.resample('2min')
.median(numeric_only=True)
.ffill()
)  

### Grouping a Date Column with pd\.Grouper

In [ ]:
print(dd
.resample('Q-JAN')
.median(numeric_only=True)
)

In [ ]:
print(dd
.reset_index()
.groupby(pd.Grouper(key='datetime', freq='Q-JAN'))
.median(numeric_only=True)
)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=c1757648-21b5-4129-a178-b9d48cb3f614' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>